In [9]:
#EE_PROJECT = "greenlight-496711"          # <-- set your Earth Engine project id

In [ ]:
"""
Sentinel-2 composites over France + Switzerland, 1-15 August, for 2023-2026.
500 m resolution, true-colour RGB, plotted as a 2x2 mosaic.

Reproduces the "DG MEME" style panel comparing August surface conditions
year over year (bare/dry vegetation appears increasingly tan/brown).

Follows the same download-then-read pattern as the fieldwork notebook
(img.getDownloadURL -> requests.get -> rasterio), which avoids the 50 MB
getInfo() cap that geemap.ee_to_numpy / sampleRectangle hits over a large AOI.
"""

import os
import io
import ee
import requests
import numpy as np
import rasterio
from rasterio.merge import merge as rio_merge
import matplotlib.pyplot as plt

# ------------------------------------------------------------------ settings
EE_PROJECT = "greenlight-496711" 
YEARS = [2023, 2024, 2025, 2026]
DATE_RANGE = ("08-01", "08-15")            # 1-15 August, each year
SCALE = 500                                # m
CRS = "EPSG:3035"                          # ETRS89 / LAEA Europe - metric, low distortion over FR+CH
MAX_CLOUD_PROB = 40                        # s2cloudless probability threshold (%)
RGB = ["B4", "B3", "B2"]
STRETCH_RANGE = (0, 3000)                  # reflectance x10000, fixed across panels
GAMMA = 1.0
OUTDIR = "S2_France_Switzerland"           # where the per-year GeoTIFFs are cached
TILE_GRID = (3, 2)                         # (nx, ny) tiles - keeps each download under EE's 48 MB cap

os.makedirs(OUTDIR, exist_ok=True)

# Area of interest: France + Switzerland (bounding box is enough at 500 m,
# swap for country outlines from a shapefile/FAO GAUL if you want a hard clip)
AOI_BBOX = [-5.5, 41.0, 10.7, 51.5]        # [lon0, lat0, lon1, lat1]
AOI = ee.Geometry.Rectangle(AOI_BBOX)

# ------------------------------------------------------------------ init
try:
    ee.Initialize(project=EE_PROJECT)
except Exception:
    ee.Authenticate()   # opens a browser login the first time; cached afterward
    ee.Initialize(project=EE_PROJECT)


# ------------------------------------------------------------------ cloud masking + composite
def s2_composite(aoi, start, end, bands):
    """Median composite of S2 L2A over aoi/[start,end], masked with s2cloudless."""
    s2 = (ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED")
            .filterBounds(aoi).filterDate(start, end))
    cld = (ee.ImageCollection("COPERNICUS/S2_CLOUD_PROBABILITY")
             .filterBounds(aoi).filterDate(start, end))

    joined = ee.ImageCollection(ee.Join.saveFirst("cloud").apply(
        primary=s2, secondary=cld,
        condition=ee.Filter.equals(leftField="system:index",
                                    rightField="system:index")))

    def mask(img):
        prob = ee.Image(img.get("cloud")).select("probability")
        return (img.updateMask(prob.lt(MAX_CLOUD_PROB))
                   .select(bands)
                   .copyProperties(img, ["system:time_start"]))

    return joined.map(mask).median().toUint16().clip(aoi)


def stretch(band):
    lo, hi = STRETCH_RANGE
    return np.clip((band - lo) / (hi - lo), 0, 1) ** GAMMA


def split_bbox(bbox, nx, ny):
    """Split a [lon0, lat0, lon1, lat1] bbox into an nx x ny grid of sub-bboxes."""
    lon0, lat0, lon1, lat1 = bbox
    lons = np.linspace(lon0, lon1, nx + 1)
    lats = np.linspace(lat0, lat1, ny + 1)
    return [(lons[i], lats[j], lons[i + 1], lats[j + 1])
            for i in range(nx) for j in range(ny)]


def download_tile(img, tile_bbox, bands, scale, crs, outfile):
    """Download one ee.Image tile as a local GeoTIFF (skips if already cached)."""
    if not os.path.exists(outfile):
        aoi = ee.Geometry.Rectangle(list(tile_bbox))
        url = img.getDownloadURL({"bands": bands, "region": aoi,
                                   "scale": scale, "crs": crs,
                                   "format": "GEO_TIFF"})
        resp = requests.get(url, timeout=600)
        resp.raise_for_status()
        with open(outfile, "wb") as f:
            f.write(resp.content)
        print(f"    saved {outfile}  ({len(resp.content)/1e6:.1f} MB)")
    else:
        print(f"    using cached {outfile}")
    return outfile


def download_composite(img, aoi_bbox, bands, scale, crs, outfile, grid=TILE_GRID):
    """Download an ee.Image as a local GeoTIFF, tiling the AOI to stay under
    Earth Engine's ~48 MB per-request cap, then mosaicking the tiles."""
    if os.path.exists(outfile):
        print(f"  using cached {outfile}")
        return outfile

    nx, ny = grid
    tiles = split_bbox(aoi_bbox, nx, ny)
    tile_files = []
    for k, t in enumerate(tiles):
        tf = outfile.replace(".tif", f"_tile{k:02d}.tif")
        download_tile(img, t, bands, scale, crs, tf)
        tile_files.append(tf)

    srcs = [rasterio.open(f) for f in tile_files]
    mosaic, transform = rio_merge(srcs)
    meta = srcs[0].meta.copy()
    meta.update({"height": mosaic.shape[1], "width": mosaic.shape[2],
                 "transform": transform})
    with rasterio.open(outfile, "w", **meta) as dst:
        dst.write(mosaic)
    for s in srcs:
        s.close()
    for f in tile_files:
        os.remove(f)

    print(f"  saved {outfile}  (mosaic of {len(tiles)} tiles)")
    return outfile


def read_rgb(tif):
    """Read RGB bands from a GeoTIFF and apply the fixed stretch."""
    with rasterio.open(tif) as src:
        arr = src.read([1, 2, 3], masked=True).astype("float32").filled(np.nan)
        extent = [src.bounds.left, src.bounds.right,
                  src.bounds.bottom, src.bounds.top]
    valid = np.isfinite(arr).all(axis=0)
    rgb = np.dstack([stretch(b) for b in arr])
    rgba = np.dstack([np.nan_to_num(rgb), valid.astype(float)])
    return rgba, extent


# ------------------------------------------------------------------ build composites
composites = {}
for year in YEARS:
    start = f"{year}-{DATE_RANGE[0]}"
    end = f"{year}-{DATE_RANGE[1]}"
    print(f"{year}: {start} -> {end}")
    img = s2_composite(AOI, start, end, RGB)
    tif = os.path.join(OUTDIR, f"S2_FR_CH_{year}_{SCALE}m.tif")
    download_composite(img, AOI_BBOX, RGB, SCALE, CRS, tif)
    rgba, extent = read_rgb(tif)
    composites[year] = (rgba, extent)
    print(f"  done, shape={rgba.shape}")

# ------------------------------------------------------------------ plot 2x2 mosaic
fig, axes = plt.subplots(2, 2, figsize=(12, 12))

for ax, year in zip(axes.flat, YEARS):
    rgba, extent = composites[year]
    ax.imshow(rgba, origin="upper", extent=extent, interpolation="nearest")
    ax.set_aspect("equal")
    ax.set_title(f"France in August: {year}", fontsize=12)
    ax.set_xticks([])
    ax.set_yticks([])

fig.suptitle(f"Sentinel-2 composites, {DATE_RANGE[0]} to {DATE_RANGE[1]} "
             f"({SCALE} m)", fontsize=14, y=0.995)
fig.tight_layout()

out = "S2_France_Switzerland_2023_2026.png"
fig.savefig(out, dpi=200, bbox_inches="tight")
plt.show()
print(f"saved {out}")

2023: 2023-08-01 -> 2023-08-15


EEException: Project 'projects/your-ee-project-id' not found or deleted.